In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# 1. Ensure the latest toolkit is installed
!pip install -U huggingface_hub


# 3. Download the specific AIO file
!hf download SeeSee21/Z-Image-Turbo-AIO z-image-turbo-fp8-aio.safetensors --local-dir /kaggle/working/ComfyUI/models/checkpoints

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 80.0 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
z-image-turbo-fp8-aio.safetensors: 100%|████| 10.3G/10.3G [00:28<00:00, 359MB/s]
✓ Downloaded
  path: /kaggle/working/ComfyUI/models/checkpoints/z-image-turbo-fp8-aio.safetensors


In [3]:
# CELL 1
!pip -q uninstall -y diffusers
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.4 MB/s eta 0:00:00


In [4]:
# CELL 2: INSTALL COMFYUI
!git clone https://github.com/comfyanonymous/ComfyUI /kaggle/working/ComfyUI
!pip -q install -r /kaggle/working/ComfyUI/requirements.txt

fatal: destination path '/kaggle/working/ComfyUI' already exists and is not an empty directory.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/kaggle/working/ComfyUI/requirements.txt'


In [5]:
# CELL 3: PLACE MODEL IN CHECKPOINTS

!ls -lh /kaggle/working/ComfyUI/models/checkpoints/

total 9.7G
-rw-r--r-- 1 root root 9.7G Apr 16 19:44 z-image-turbo-fp8-aio.safetensors


In [6]:
# CELL 4: DOWNLOAD A SIMPLE Z-IMAGE WORKFLOW JSON
!mkdir -p /kaggle/working/ComfyUI/user/default/workflows

In [7]:
# CELL 3: verify GPU torch
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
print("gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

torch: 2.10.0+cu128
cuda available: True
torch cuda version: 12.8
gpu name: Tesla T4


In [8]:
# CELL 5: RUN COMFYUI
#%cd /kaggle/working/ComfyUI
#!python main.py --listen 0.0.0.0 --port 8188

In [9]:
import os
import subprocess

# 1. Clean up the old broken files
!rm -rf /kaggle/working/zrok
os.makedirs("/kaggle/working/zrok", exist_ok=True)

# 2. Download and Extract zrok v2 (The new version)
print("📥 Downloading zrok v2.0...")
# This pulls the compressed archive and extracts it directly into the folder
!wget -q https://github.com/openziti/zrok/releases/download/v2.0.1/zrok_2.0.1_linux_amd64.tar.gz -O /kaggle/working/zrok/zrok.tar.gz
!tar -xzf /kaggle/working/zrok/zrok.tar.gz -C /kaggle/working/zrok/
!rm /kaggle/working/zrok/zrok.tar.gz

# 3. Verify and Set Permissions
# Note: In v2, the binary is sometimes named 'zrok2'
ZROK_PATH = "/kaggle/working/zrok/zrok2" 
if not os.path.exists(ZROK_PATH):
    ZROK_PATH = "/kaggle/working/zrok/zrok" # Fallback check

if os.path.exists(ZROK_PATH) and os.path.getsize(ZROK_PATH) > 0:
    os.chmod(ZROK_PATH, 0o755)
    print(f"✅ Success! zrok is now {os.path.getsize(ZROK_PATH) / 1e6:.2f} MB")
else:
    print("❌ ERROR: Download failed again. Check if 'Internet' is enabled in Kaggle Settings!")

📥 Downloading zrok v2.0...
✅ Success! zrok is now 95.57 MB


In [10]:
import os
import subprocess
import threading
import time
import re

# --- CONFIGURATION ---
ZROK_TOKEN = "R5nMRv9R2zgn"  # <-- Replace this with your actual token
PORT = 8188
COMFY_PATH = "/kaggle/working/ComfyUI"
ZROK_BIN = "/kaggle/working/zrok/zrok2" # v2 path

# 1. Move your AIO model to the right place if you haven't yet
if os.path.exists("/kaggle/working/models/z-image-turbo-fp8-aio.safetensors"):
    print("🚚 Moving model to ComfyUI checkpoints...")
    os.makedirs(f"{COMFY_PATH}/models/checkpoints", exist_ok=True)
    subprocess.run(f"mv /kaggle/working/models/z-image-turbo-fp8-aio.safetensors {COMFY_PATH}/models/checkpoints/", shell=True)

# 2. Enable zrok environment
print("🔑 Enabling zrok environment...")
subprocess.run(f"{ZROK_BIN} enable {ZROK_TOKEN}", shell=True, capture_output=True)

# 3. Start Tunnel in Background
def start_zrok():
    print(f"🚀 Starting Proxy Tunnel...")
    # --headless is required in v2 to print the link in Kaggle logs
    cmd = f"{ZROK_BIN} share public http://127.0.0.1:{PORT} --headless"
    with subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            # Detect and print the public URL
            if "https://" in line:
                match = re.search(r"https://[a-zA-Z0-9.-]+\.share\.zrok\.io", line)
                if match:
                    url = match.group(0)
                    print("\n" + "⭐" * 50)
                    print(f"🔗 COMFYUI IS LIVE AT: {url}")
                    print("⭐" * 50 + "\n")

threading.Thread(target=start_zrok, daemon=True).start()

# 4. Launch ComfyUI
time.sleep(5)
if os.path.exists(COMFY_PATH):
    print("🎨 Launching ComfyUI Server...")
    os.chdir(COMFY_PATH)
    # Optimized for 2026 T4 x2 Environment
    !python main.py --use-pytorch-cross-attention --preview-method auto --dont-print-server --fp16-vae
else:
    print(f"❌ Error: Could not find ComfyUI at {COMFY_PATH}")

🔑 Enabling zrok environment...
🚀 Starting Proxy Tunnel...
🎨 Launching ComfyUI Server...
python3: can't open file '/kaggle/working/ComfyUI/main.py': [Errno 2] No such file or directory


In [11]:
!nvidia-smi

Thu Apr 16 19:45:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

link api

In [12]:
import os
import subprocess
import threading
import time
import re

# --- CONFIG ---
ZROK_TOKEN = "R5nMRv9R2zgn" 
PORT = 8188
COMFY_PATH = "/kaggle/working/ComfyUI"
ZROK_BIN = "/kaggle/working/zrok/zrok2"

# 1. Enable zrok
subprocess.run(f"{ZROK_BIN} enable {ZROK_TOKEN}", shell=True, capture_output=True)

# 2. Tunnel Thread
def start_tunnel():
    print(f"🚀 Starting Tunnel...")
    cmd = f"{ZROK_BIN} share public http://127.0.0.1:{PORT} --headless"
    with subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            if "https://" in line:
                match = re.search(r"https://[a-zA-Z0-9.-]+\.share\.zrok\.io", line)
                if match:
                    print(f"\n⭐ YOUR LINK: {match.group(0)} ⭐\n")

threading.Thread(target=start_tunnel, daemon=True).start()

# 3. Launch ComfyUI with Extreme RAM Safety
time.sleep(5)
if os.path.exists(COMFY_PATH):
    print("🎨 Launching ComfyUI (Extreme Safety Mode)...")
    os.chdir(COMFY_PATH)
    
    # --cpu-vae: Stops the GPU from "choking" during the final render
    # --lowvram: Keeps the 9GB model from exploding the VRAM
    # --use-split-cross-attention: Essential for keeping the 30GB System RAM stable
    !python main.py \
        --use-split-cross-attention \
        --preview-method auto \
        --dont-print-server \
        --cpu-vae \
        --lowvram \
        --cuda-device 0

🚀 Starting Tunnel...
🎨 Launching ComfyUI (Extreme Safety Mode)...
python3: can't open file '/kaggle/working/ComfyUI/main.py': [Errno 2] No such file or directory
